In [2]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# 1. Load Master Cleaned Dataset
df = pd.read_csv("college_majors_bachelors_cleaned.csv")

# 2. Define Target, Features, and Grouping ID
target_col = 'median_earnings_4yr_usd'

exclude_cols = [
    'median_earnings_4yr_usd', 'log_median_earnings_4yr_usd', 'is_high_roi', 
    'is_high_debt_risk', 'median_earnings_5yr_usd', 'earnings_growth_pct_1yr_to_5yr',
    'count_working_in_state_5yr', 'pct_working_in_state_5yr', 'ai_tool_examples',
    'unitid', 'opeid', 'opeid6', 'program_id', 'institution_name'
]

feature_cols = [c for c in df.columns if c not in exclude_cols]

X = df[feature_cols]
y = df[target_col]
groups = df['opeid6']

# 3. Institution-Grouped Train/Test Split (Preventing Data Leakage)
gss = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

# Drop opeid6 from features so it is not used as an explicit predictor
if 'opeid6' in X_train.columns:
    X_train = X_train.drop(columns=['opeid6'])
    X_test = X_test.drop(columns=['opeid6'])

# 4. Identify Numeric and Categorical Columns
numeric_features = X_train.select_dtypes(include=[np.number]).columns.tolist()
categorical_features = X_train.select_dtypes(exclude=[np.number]).columns.tolist()

# 5. Preprocessing Pipelines (Fixed with sparse_output=False for HistGradientBoosting compatibility)
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='Missing')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', min_frequency=20, sparse_output=False))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ]
)
# 6. Define the Three Supervised Models
models = {
    'Ridge Regression (Linear)': Pipeline(steps=[('prep', preprocessor), ('model', Ridge(alpha=10.0))]),
    'Random Forest Regressor': Pipeline(steps=[('prep', preprocessor), ('model', RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1))]),
    'Histogram Gradient Boosting': Pipeline(steps=[('prep', preprocessor), ('model', HistGradientBoostingRegressor(random_state=42))])
}

# 7. Train, Predict, and Evaluate All Models
results = []

for name, pipeline in models.items():
    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)
    
    results.append({
        'Model Family': name,
        'Test MAE ($USD)': mean_absolute_error(y_test, y_pred),
        'Test RMSE ($USD)': np.sqrt(mean_squared_error(y_test, y_pred)),
        'Test R-squared': r2_score(y_test, y_pred)
    })

# 8. Output Summary Table
results_df = pd.DataFrame(results).sort_values(by='Test MAE ($USD)')
print(results_df.to_string(index=False))

               Model Family  Test MAE ($USD)  Test RMSE ($USD)  Test R-squared
    Random Forest Regressor       269.641017       1872.855758        0.991625
Histogram Gradient Boosting       523.470667       2210.363762        0.988335
  Ridge Regression (Linear)      1904.558214       3206.760694        0.975447


First Notes: 
- Red Flag — Likely Target Leakage (R^2 = 0.9916, MAE = $269): An R^2 near 99 % and an average error of only ~$270 on $60,000+ salary data is almost certainly "too good to be true." This strongly suggests Target Leakage in X — most likely raw median_earnings_4yr_usd was left in the feature matrix while predicting log_median_earnings_4yr_usd, allowing the model to simply calculate y = log(X).
- Tree Ensembles Dominate Linear Models: Both non-linear tree models drastically beat Ridge Regression (MAE $1,904 vs. $269/$523). Trees easily fit complex non-linear mathematical relationships that linear lines cannot handle.
- Random Forest Edge over Gradient Boosting: Random Forest achieved nearly half the error of Histogram Gradient Boosting ($269.64 vs. $523.47), showing its bagging strategy smoothed out predictions exceptionally well.
- Action Item: Print X_train.columns to verify that median_earnings_4yr_usd, is_high_roi, is_high_debt_risk, and all 1-year salary columns were 100% removed from X. 

In [4]:
# 1. Determine feature columns from 'X' or 'df'
feature_cols = X.columns if 'X' in locals() else df.columns

print("Checking feature columns for target leakage:")

# 2. Check for any target or outcome-related keywords
leaky_cols = [
    col for col in feature_cols 
    if any(keyword in col.lower() for keyword in ['earnings', 'roi', 'debt', 'payment', '1yr', '5yr'])
]

if leaky_cols:
    print("\n⚠️ FOUND LEAKY COLUMNS IN YOUR FEATURES:")
    for col in leaky_cols:
        print(f" - {col}")
else:
    print("\n✅ No obvious leaky salary/debt columns found in features.")

Checking feature columns for target leakage:

⚠️ FOUND LEAKY COLUMNS IN YOUR FEATURES:
 - earnings_cohort_size_4yr
 - national_median_earnings_4yr_usd
 - national_p25_earnings_4yr_usd
 - national_p75_earnings_4yr_usd
 - earnings_vs_national_pct
 - earnings_cohort_size_1yr
 - earnings_cohort_size_5yr
 - not_working_count_5yr
 - count_above_hs_threshold_5yr
 - median_debt_usd
 - debt_borrower_count
 - median_monthly_payment_usd
 - earnings_trajectory_category
 - debt_to_earnings_1yr
 - debt_to_earnings_4yr
 - payment_to_income_pct_1yr
 - pct_above_hs_threshold_5yr
 - pct_working_5yr
 - earnings_4yr_status
 - earnings_1yr_status
 - earnings_5yr_status
 - debt_status


Even if median_earnings_4yr_usd itself was removed, the decision trees discovered two mathematical formulas hiding right inside your feature matrix:
- The Division Leak (debt_to_earnings_4yr + median_debt_usd): Debt_to_earnings_4yrs = (Median_debt_usd)/ (median_earnings_4yr_usd)
This allowed the trees to reverse-engineer the exact 4-year salary down to the penny. 
- The Multiplication Leak (earnings_vs_national_pct + national_median_earnings_4yr_usd): earnings_vs_national_pct = (Median_earnings_4yr_usd)/(national_median_earnings_4yr_usd). 
Multiplying those two columns reconstructs the exact target salary instantly.

In [6]:
# Complete list of post-graduation target leakage columns to purge from X
leakage_cols_to_drop = [
    # Direct target & salary columns
    'median_earnings_4yr_usd',
    'log_median_earnings_4yr_usd',
    'median_earnings_1yr_usd',
    'median_earnings_5yr_usd',
    'earnings_growth_pct_1yr_to_5yr',
    
    # Mathematical leaks (National benchmarks & trajectory categories)
    'earnings_vs_national_pct',
    'national_median_earnings_4yr_usd',
    'national_p25_earnings_4yr_usd',
    'national_p75_earnings_4yr_usd',
    'earnings_trajectory_category',
    'earnings_4yr_status',
    'earnings_1yr_status',
    'earnings_5yr_status',
    
    # Mathematical leaks (Ratios containing salary in the denominator!)
    'debt_to_earnings_1yr',
    'debt_to_earnings_4yr',
    'payment_to_income_pct_1yr',
    
    # Engineered classification targets
    'is_high_roi',
    'is_high_debt_risk',
    
    # Post-graduation cohort tracking & status flags
    'earnings_cohort_size_4yr',
    'earnings_cohort_size_1yr',
    'earnings_cohort_size_5yr',
    'not_working_count_5yr',
    'count_above_hs_threshold_5yr',
    'pct_above_hs_threshold_5yr',
    'pct_working_5yr',
    'debt_status'
]

# Clean feature matrix containing ONLY pre-graduation predictors (SAT, tuition, field, labor stats)
X = df.drop(columns=[c for c in leakage_cols_to_drop if c in df.columns])
y = df['log_median_earnings_4yr_usd']

In [8]:
train_idx, test_idx = next(gss.split(X, y, groups=df['opeid6']))
X_train_raw, X_test_raw = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

# Transform with pipeline
X_train = preprocessor.fit_transform(X_train_raw)
X_test = preprocessor.transform(X_test_raw)

ValueError: Some column names are not columns of the dataframe: {'pct_above_hs_threshold_5yr', 'national_p75_earnings_4yr_usd', 'national_median_earnings_4yr_usd', 'not_working_count_5yr', 'count_above_hs_threshold_5yr', 'earnings_vs_national_pct', 'payment_to_income_pct_1yr', 'debt_to_earnings_1yr', 'debt_to_earnings_4yr', 'national_p25_earnings_4yr_usd', 'earnings_cohort_size_4yr', 'earnings_cohort_size_1yr', 'earnings_cohort_size_5yr', 'pct_working_5yr'}